import all necessary libraries

In [0]:
import pyspark.sql.functions as F

check the external location

In [0]:
%sql
describe external location `control_datalake_raw`

Extract the URL

In [0]:
raw_dir = spark.sql("describe external location `control_datalake_raw`").select("url").collect()[0][0]
#display(base_dir)

Extract the data from containers to variables

In [0]:
# set the path to the raw data
synthetic_data_dir = raw_dir+"/synthetic_data.json"
data_overview_dir = raw_dir+"/data_overview.json"
data_dictionary_dir = raw_dir+"/data_dictionary.json"
data_anomaly_dir = raw_dir+"/data_anomaly.json"

#assign the raw data to variables 
data_overview = spark.read.format("json").load(data_overview_dir)
data_dictionary = spark.read.format("json").load(data_dictionary_dir)
data_anomaly = spark.read.format("json").load(data_anomaly_dir)
synthetic_data = spark.read.format("json").load(synthetic_data_dir)


Create tables

In [0]:
synthetic_data = synthetic_data.withColumnRenamed("_id","id").withColumnRenamed("Transaction Timestamp","timestamp").withColumnRenamed("Usage Amount","usage_amount").withColumnRenamed("User ID","user_id").withColumnRenamed("Account Number","account_number").withColumnRenamed("Registration Date","registration_date").withColumnRenamed("Status","status").withColumnRenamed("Email","email").withColumnRenamed("Name","name").withColumnRenamed("Phone","phone")
display(synthetic_data)

In [0]:
catalog = "control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")
#write the data to the database
data_overview.write.mode("overwrite").saveAsTable("raw_data_overview")
data_dictionary.write.mode("overwrite").saveAsTable("raw_data_dictionary")
data_anomaly.write.mode("overwrite").saveAsTable("raw_data_anomaly")
synthetic_data.write.mode("overwrite").saveAsTable("raw_synthetic_data")



create control logic exception table

In [0]:
%sql
USE raw_control_datalake.dev;
  

In [0]:
df =spark.read.table("raw_data_anomaly")
display(df)

In [0]:
catalog = "raw_control_datalake"
database = "dev"
spark.sql(f"use {catalog}.{database}")


In [0]:
synthetic_df = spark.read.table("raw_synthetic_data").filter("status='Suspended'")
display(synthetic_df)


In [0]:
%sql
--SQL version of locating the exceptions
SELECT *
FROM raw_synthetic_data
WHERE status = 'Suspended';

In [0]:
# Convert it into python

control_logic = f"""SELECT *
                    FROM raw_synthetic_data
                    WHERE status = 'Suspended';
                    """

In [0]:
print(control_logic)

In [0]:
spark.sql(control_logic).display()

In [0]:
spark.sql(control_logic).display()

create a control logic table

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.createDataFrame([
    (1, control_logic, "Check suspended customers", "ACTIVE")
], ["Reference_number","Control_logic","Control_logic_description","Control_logic_status"])

df = df.withColumn("created_timestamp", current_timestamp())

df.write.mode("append").saveAsTable(f"{catalog}.{database}.raw_control_logic")